In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler  # SVM 和 KNN 建议标准化

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

'''
    get_failure_mutipro.py和get_normal_mutipro.py
    在2020-2025这6年的数据中,一共选出来了20160个故障样本。
    其中最多的型号是ST12000NM0008,为2328个。
    相应地, 在所有csv文件中抽取了型号为ST12000NM0008的2328个正常样本。
'''

## 读取数据
normal = pd.read_csv("./normal.csv")
failure_raw = pd.read_csv("./failure.csv")
failure = failure_raw[failure_raw["model"] == "ST12000NM0008"]
print("normal hdds:", len(normal))
print("failure hdds:", len(failure))
df_data = pd.concat([normal, failure], ignore_index=True) # 合并

normal hdds: 2328
failure hdds: 2328


In [8]:
## 原始特征
features_specified = []
features = [1,2,3,4,5,7,8,9,10,11,12,13,15,16,17,18,22,23,24,168,170,173,174,175,177,179,
            180,181,182,183,184,187,188,189,190,191,192,193,194,195,196,197,198,199,200,
            201,202,206,210,218,220,222,223,224,225,226,231,232,233,234,235,240,241,242,
            245,247,248,250,251,252,254,255,27,71,82,90,160,161,163,164,165,166,167,169,
            171,172,176,178,230,244,246,211,212]
# features = [5, 9, 187, 188, 193, 194, 197, 198, 241, 242]
for feature in features:
    features_specified += ["smart_{0}_raw".format(feature)]
features_specified += ["failure"]
## 数据
df_data = df_data[features_specified]

In [ ]:
# ## 保存一份供matlab用
# df_data.fillna(0)
# df_data.to_csv("concat.csv")

In [12]:
## 检查全空值得列
# 1. 检查每列的所有值是否都是空值（返回布尔值的Series，True表示该列全为NaN）
all_nan_columns = df_data.isnull().all(axis=0)
# 2. 统计True的数量（即全为空值列的数量）
nan_columns_count = all_nan_columns.sum()
# 3. 输出结果
print(f"\n所有行都是空值（NaN）的列的数量：{nan_columns_count}")

# （可选）输出具体是哪些列全为空值，方便后续处理
nan_column_names = all_nan_columns[all_nan_columns].index.tolist()
print(f"全为空值的列名：{nan_column_names}")

non_all_nan_columns = all_nan_columns[~all_nan_columns].index.tolist()
print(f"\n不是所有行都是空值的列的数量：{len(non_all_nan_columns)}")
print("不是所有行都是空值的列名列表：")
for idx, col_name in enumerate(non_all_nan_columns, 1):
    print(f"{idx}. {col_name}")


所有行都是空值（NaN）的列的数量：70
全为空值的列名：['smart_2_raw', 'smart_8_raw', 'smart_11_raw', 'smart_13_raw', 'smart_15_raw', 'smart_16_raw', 'smart_17_raw', 'smart_22_raw', 'smart_23_raw', 'smart_24_raw', 'smart_168_raw', 'smart_170_raw', 'smart_173_raw', 'smart_174_raw', 'smart_175_raw', 'smart_177_raw', 'smart_179_raw', 'smart_180_raw', 'smart_181_raw', 'smart_182_raw', 'smart_183_raw', 'smart_184_raw', 'smart_189_raw', 'smart_191_raw', 'smart_196_raw', 'smart_201_raw', 'smart_202_raw', 'smart_206_raw', 'smart_210_raw', 'smart_218_raw', 'smart_220_raw', 'smart_222_raw', 'smart_223_raw', 'smart_224_raw', 'smart_225_raw', 'smart_226_raw', 'smart_231_raw', 'smart_232_raw', 'smart_233_raw', 'smart_234_raw', 'smart_235_raw', 'smart_245_raw', 'smart_247_raw', 'smart_248_raw', 'smart_250_raw', 'smart_251_raw', 'smart_252_raw', 'smart_254_raw', 'smart_255_raw', 'smart_27_raw', 'smart_71_raw', 'smart_82_raw', 'smart_90_raw', 'smart_160_raw', 'smart_161_raw', 'smart_163_raw', 'smart_164_raw', 'smart_165_raw',

In [10]:
len(df_data.columns)

94

In [ ]:
X_data = df_data[features_specified] # 数据
Y_data = df_data['failure'] # 标签
X_data.isnull().sum() # 查看每列有多少空值
X_data = X_data.fillna(0) # 用0填充空值

print("normal hdds:", len(Y_data) - np.sum(Y_data.values))
print("failure hdds:", np.sum(Y_data.values))

X_train, X_test, Y_train, Y_test = train_test_split(X_data, Y_data, test_size=0.2, random_state=0)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 定义模型
classifiers = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=0,
        n_jobs=-1
    ),
    
    "LightGBM": LGBMClassifier(
        random_state=0,
        n_estimators=300,
        learning_rate=0.03
    ),
    
    "CatBoost": CatBoostClassifier(
        random_seed=0,
        iterations=300,
        learning_rate=0.03,
        depth=6,
        verbose=0
    ),
    
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=0,
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3
    ),
    
    "AdaBoost": AdaBoostClassifier(
        n_estimators=300,
        learning_rate=0.03,
        random_state=0
    )
}

# 存储结果
results = {}

# 训练与评估
for name, clf in classifiers.items():
    print(f"\n=== Training {name} ===")
    
    clf.fit(X_train, Y_train)
    Y_pred = clf.predict(X_test)
    
    acc = accuracy_score(Y_test, Y_pred)
    prec = precision_score(Y_test, Y_pred, zero_division=0)
    rec = recall_score(Y_test, Y_pred, zero_division=0)
    f1 = f1_score(Y_test, Y_pred, zero_division=0)
    
    results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    }
    
    print(f"{name} - Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")

In [ ]:
## RF训练测试
rfc = RandomForestClassifier()
rfc.fit(X_train, Y_train)
Y_pred = rfc.predict(X_test) 
acc = accuracy_score(Y_test, Y_pred)
prec = precision_score(Y_test, Y_pred, zero_division=0)
rec = recall_score(Y_test, Y_pred, zero_division=0)
f1 = f1_score(Y_test, Y_pred, zero_division=0)
print(f" 分类结果: \n acc: {acc}\n pre: {prec} \n recall: {rec} \n f1: {f1}")

# 混淆矩阵
cm = confusion_matrix(Y_test, Y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Valid', 'Failed'], 
            yticklabels=['Valid', 'Failed'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# ===================== 新增：特征重要性计算 + 可视化 =====================
## 1. 计算特征重要性并输出数值
feature_importance = rfc.feature_importances_
# 构建特征名-重要性的字典并排序
feature_importance_dict = dict(zip(features_specified, feature_importance))
# 按重要性从高到低排序
sorted_importance = sorted(feature_importance_dict.items(), key=lambda x: x[1], reverse=True)

# 打印特征重要性（保留4位小数，便于论文展示）
print("\n=== 特征重要性（从高到低）===")
for idx, (feature, importance) in enumerate(sorted_importance, 1):
    print(f"{idx}. {feature}: {importance:.4f}")

## 2. 绘制特征重要性柱状图（论文级可视化）
# 提取排序后的特征名和重要性值
sorted_features = [item[0] for item in sorted_importance]
sorted_values = [item[1] for item in sorted_importance]

# 设置画布大小（适合论文排版）
plt.figure(figsize=(10, 6))
# 绘制柱状图（颜色渐变，更美观）
colors = sns.color_palette('Blues_r', len(sorted_features))
bars = plt.bar(sorted_features, sorted_values, color=colors)

# 添加数值标签（显示在柱子上方）
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{height:.4f}', ha='center', va='bottom', fontsize=9)

# 设置图表样式（符合论文规范）
plt.title('Feature Importance of Random Forest', fontsize=12, fontweight='bold')
plt.xlabel('Features', fontsize=10)
plt.ylabel('Importance', fontsize=10)
# 旋转x轴标签，避免重叠
plt.xticks(rotation=45, ha='right')
# 去除顶部和右侧边框
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
# 调整布局，避免标签被截断
plt.tight_layout()
# 显示图表
plt.show()